In [1]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
from graph_utils import get_graph_embeddings_from_string_with_model, get_bilstm_embeddings_from_string_with_model, get_token_bilstm_embeddings_from_string_with_model, get_adapter_embeddings_from_string_with_model, make_graph_ready_for_token_ids
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
from eval_utils import ensure_in_seq_string_form

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
device_name = 'cuda:2'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

GuidanceAdapter(
  (proj): Linear(in_features=1024, out_features=512, bias=True)
)

In [4]:
s1 = ['G:7', 'C:maj']
s2 = ['D:min7','G:7', 'C:maj']

In [5]:
st1 = ensure_in_seq_string_form(s1)
st2 = ensure_in_seq_string_form(s2)

print(st1)
print(st2)

b_G:7_@2C:maj_@2
b_D:min7_@2G:7_@2b_C:maj_@2


In [11]:
y_graph_s1 = get_graph_embeddings_from_string_with_model(st1, graph_adapter_model)
y_graph_s2 = get_graph_embeddings_from_string_with_model(st2, graph_adapter_model)

y_token_s1 = get_token_bilstm_embeddings_from_string_with_model(st1, token_adapter_model)
y_token_s2 = get_token_bilstm_embeddings_from_string_with_model(st2, token_adapter_model)

In [13]:
print(torch.abs(y_graph_s1 - y_graph_s2).sum().item())
print(torch.abs(y_token_s1 - y_token_s2).sum().item())

140.66055297851562
102.96466064453125
